# Stage 1 — Data Ingestion

**Adidas Global Catalogue 2026** · Path A (Descriptive) + B (Predictive)
Governed by [`DOCS/STRUCTURE.md`](../DOCS/STRUCTURE.md) Stage 1 · follows Stage 0
([`00_problem_framing.md`](00_problem_framing.md), [`00_ghost_deck.md`](00_ghost_deck.md)).

**Goal:** acquire the raw catalogue into the analysis environment *without modifying it*, and produce an
auditable ingestion record — source hash, row/column counts, first-load schema validation, and a
master-vs-country-file reconciliation.

> **Immutability (STRUCTURE Stage 1):** the raw layer is `archive/` and is **never edited**. All typing,
> cleaning, and feature work happens in Stage 2. This notebook is read-only over the raw data.

All logic lives in [`src/ingestion.py`](../src/ingestion.py); this notebook orchestrates and narrates it.

In [1]:
import sys
from pathlib import Path

# Make src/ importable and locate the project root regardless of the kernel's CWD.
ROOT = Path.cwd()
while not (ROOT / "src" / "ingestion.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import ingestion  # src/ingestion.py

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
print("Project root:", ROOT)

Project root: C:\Users\HP ENVY\OneDrive\Documents\Personal Project\Dataset\Adidas Global Catalogue 2026


## 1.1 — Acquire & fingerprint the source

We compute a SHA-256 over the raw master so any future change to the file is detectable, then load it
exactly as stored (pandas' own type inference, no coercion) so Stage 2 can see the raw dtypes.

In [2]:
df, report = ingestion.ingest()

print("Source        :", report.source)
print("Ingested (UTC):", report.ingested_at_utc)
print("SHA-256       :", report.sha256)
print(f"Shape         : {report.n_rows:,} rows x {report.n_cols} cols")
print("Structure OK  :", report.schema_ok)

Source        : archive\Adidas_Global.csv
Ingested (UTC): 2026-07-12T13:19:14+00:00
SHA-256       : 97ed5637b94a163f5dbdfbd052574da6d5986829f4d70e9653208e7a47532fec
Shape         : 44,888 rows x 35 cols
Structure OK  : True


## 1.2 — First-load schema validation

The schema contract (`src/ingestion.py :: validate_schema`) confirms **structure** (all 35 columns
present; `country_code`, `snapshot_date`, `currency` within their expected domains) and **warns** on
known value-level defects that belong to Stage 2 — it does not fix them here.

In [3]:
for line in report.checks:
    print(line)

✅ All 35 expected columns present.
✅ country_code: 10 markets, all within expected set.
✅ snapshot_date: 2 snapshot dates, all within expected set.
✅ currency: 9 currencies, all within expected set.
⚠️  discount_pct: 711 negative values (known defect — fix in Stage 2/§3).
⚠️  rating_count: 8,338 negative values (known defect — fix in Stage 2/§3).
⚠️  Reconciliation: 7/10 markets match their country file exactly.


**Read-out.** Structure is intact — 35 columns, 10 markets, 2 snapshot dates, 9 currencies (EUR is
shared by DE + FR, so 9 currencies across 10 markets is expected). The two warnings are exactly the
defects the data assessment predicted:

- `rating_count`: **8,338** rows carry the `-99` fake-missing sentinel (§3 defect #1).
- `discount_pct`: **711** rows are negative — a "discount" that is actually a markup (§3 defect #6).

Both are logged now and repaired in Stage 2 (`rating_count_clean`, `discount_pct_clean`).

## 1.3 — Reconciliation: is the master a faithful concatenation?

We treat `archive/Adidas_Global.csv` as the single source of truth, but must know whether it equals the
sum of the 10 country files or was filtered/deduped upstream. We compare per-market row counts.

In [4]:
recon = report.reconciliation
recon

,market,master_rows,country_file_rows,delta,match
0,BR,4680,4680,0,True
1,CN,904,904,0,True
2,DE,5004,5004,0,True
3,FR,4061,4061,0,True
4,GB,4283,5059,-776,False
5,IN,6643,6658,-15,False
6,JP,6037,6037,0,True
7,KR,2538,2538,0,True
8,MX,4992,4992,0,True
9,US,5746,5936,-190,False


In [5]:
mismatch = recon.loc[~recon["match"]]
total_gap = int(recon["delta"].fillna(0).sum())
print(f"Markets matching exactly : {int(recon['match'].sum())}/10")
print(f"Markets with a gap       : {list(mismatch['market'])}")
print(f"Net master-minus-country : {total_gap:,} rows")

Markets matching exactly : 7/10
Markets with a gap       : ['GB', 'IN', 'US']
Net master-minus-country : -981 rows


> **⚠️ Ingestion finding (documented, not patched).** The master is a faithful concatenation for
> **7 of 10** markets. For **GB (−776), IN (−15), US (−190)** the master holds *fewer* rows than the
> standalone country file — **981 fewer rows in total**. The master was evidently filtered or deduped
> upstream for these three markets.
>
> **Decision (per IMPLEMENTATION_PLAN §1):** the master remains the single source of truth — we do **not**
> back-fill from the country files, because mixing two extraction passes would double-count and blur the
> grain. We record the gap so any GB/US/IN-specific finding is read with this caveat, and so a later pass
> can investigate the upstream filter if a market-level count becomes decision-critical.

## 1.4 — Raw dtypes & preview (input to the Stage 2 casting plan)

Stage 1 does not cast types; it *documents* them so Stage 2 has an explicit plan. Note `snapshot_date`
(string → datetime), `in_stock` (object of `true`/`false` strings → bool), and the numeric columns that
still carry sentinels/negatives.

In [6]:
dtypes = (df.dtypes.astype(str).rename("raw_dtype").to_frame()
          .assign(n_missing=df.isna().sum(),
                  pct_missing=(df.isna().mean() * 100).round(1),
                  n_unique=df.nunique(dropna=True)))
dtypes

,raw_dtype,n_missing,pct_missing,n_unique
snapshot_date,str,0,0.0,2
country_code,str,0,0.0,10
product_name,str,0,0.0,2051
model_number,str,351,0.8,1492
currency,str,0,0.0,9
price_local,float64,0,0.0,431
sale_price_local,float64,18918,42.1,392
gender_segment,str,366,0.8,9
size_label,str,0,0.0,962
category,str,517,1.2,53


In [7]:
# A narrow, human-readable preview of identity + the four levers (price/assort/stock/discount).
preview_cols = ["snapshot_date", "country_code", "currency", "product_name",
                "base_model_number", "color_name", "category", "price_local",
                "sale_price_local", "discount_pct", "size_count",
                "available_size_count", "availability_status", "in_stock",
                "rating", "rating_count"]
df[preview_cols].head(8)

,snapshot_date,country_code,currency,product_name,base_model_number,color_name,category,price_local,sale_price_local,discount_pct,size_count,available_size_count,availability_status,in_stock,rating,rating_count
0,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,IN_STOCK,True,NaN,-99.0
1,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
2,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
3,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
4,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
5,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
6,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0
7,2026-03-09,BR,BRL,AGASALHO COM CAPUZ BIG LOGO,KSG46,NaN,Lifestyle,429.99,549.99,0.0,26,6.0,NOT_AVAILABLE,True,NaN,-99.0


## 1.5 — Persist the ingestion record

We write the full report to `reports/stage1_ingestion_report.md` as durable Stage 1 gate evidence
(source, hash, timestamp, checks, reconciliation). The raw file itself is left untouched.

In [8]:
reports_dir = ROOT / "reports"
reports_dir.mkdir(exist_ok=True)
out = reports_dir / "stage1_ingestion_report.md"
out.write_text(report.to_markdown(), encoding="utf-8")
print("Wrote", out.relative_to(ROOT))

Wrote reports\stage1_ingestion_report.md


## Stage 1 — Gate Checklist

- [x] **Raw data in an immutable layer** — `archive/` treated as `data/raw/`, never edited.
- [x] **Source, timestamp, row/column counts logged** — plus a SHA-256 fingerprint.
- [x] **Schema validated on first load** — 35 columns; markets/dates/currencies within expected domains.
- [x] **Discrepancies documented, not silently fixed** — `-99` sentinel (8,338), negative `discount_pct`
      (711), and the GB/IN/US reconciliation gap (−981 rows) are all recorded for Stage 2.
- [x] **PII / governance** — public product-catalogue data, no personal identifiers; no special handling.

### Findings carried into Stage 2
1. `rating_count == -99` → `rating_count_clean` (NaN). *(§3 #1)*
2. `discount_pct` negatives + `sale_price` / `discount_pct=0` inconsistencies → recompute `discount_pct_clean`. *(§3 #6, #7)*
3. Cast `snapshot_date`→datetime, `in_stock`→bool, `category*`/`gender_segment`→category.
4. **Reconciliation caveat:** GB/US/IN master counts are short of the country files — flag on any market-level readout.

**→ Gate passed. Proceed to Stage 2 — Cleaning & Feature Derivation.**